# NVD CPE Semantic Embedding Pipeline (Memory-Safe Edition)

**Problem:** The original notebook crashes on Colab free tier because it tries to hold the entire 2.4 GB embedding matrix + DataFrame + raw data in RAM at once.

**Solution:** This notebook processes CPEs in **chunks**, embeds batch-by-batch, and writes to Parquet **incrementally** using PyArrow. Peak RAM stays under ~6 GB.

**Data source:** `mirror.cveb.in` — `nvdcpe-2.0.tar.gz` (~76 MB)

**Output:** `cpe_embeddings.parquet` (~2.8 GB)

**Chunk strategy:**
- Download & extract: all at once (~400 MB RAM)
- Build DataFrame: all at once (~250 MB RAM)
- **Embed & write: 20,000 CPEs at a time** (~80 MB embeddings per chunk)
- Clear memory between chunks


In [ ]:

!pip install -q sentence-transformers pandas pyarrow tqdm requests

import os, json, time, tarfile, io, gc
import requests
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import pyarrow as pa
import pyarrow.parquet as pq

print("All dependencies installed")


✅ All dependencies installed


In [ ]:


MIRROR_URL = "https://mirror.cveb.in/nvd/json/cpe/2.0/nvdcpe-2.0.tar.gz"
MODEL_NAME = "all-MiniLM-L6-v2"
EMBED_BATCH_SIZE = 256       # inference batch size (lower = less GPU RAM)
CHUNK_SIZE = 20000           # CPEs to embed & write per chunk (keeps RAM low)
OUTPUT_FILE = "cpe_embeddings.parquet"

print("Config loaded")
print(f"   Mirror URL: {MIRROR_URL}")
print(f"   Model: {MODEL_NAME}")
print(f"   Embed batch size: {EMBED_BATCH_SIZE}")
print(f"   Chunk size: {CHUNK_SIZE} CPEs per write cycle")


🔧 Config loaded
   Mirror URL: https://mirror.cveb.in/nvd/json/cpe/2.0/nvdcpe-2.0.tar.gz
   Model: all-MiniLM-L6-v2
   Embed batch size: 256
   Chunk size: 20000 CPEs per write cycle


In [ ]:


def download_and_extract_cpe_dictionary(url: str):
    print(f"Downloading from: {url}")
    resp = requests.get(url, stream=True, timeout=120)
    resp.raise_for_status()

    total = int(resp.headers.get("content-length", 0))
    block_size = 1024 * 1024
    buffer = io.BytesIO()

    with tqdm(total=total, unit="B", unit_scale=True, desc="Downloading tar.gz") as t:
        for chunk in resp.iter_content(chunk_size=block_size):
            if chunk:
                buffer.write(chunk)
                t.update(len(chunk))

    buffer.seek(0)
    size_mb = buffer.getbuffer().nbytes / (1024 * 1024)
    print(f"Downloaded: {size_mb:.1f} MB")

    print("Extracting tar.gz and parsing JSON chunks...")
    records = []
    with tarfile.open(fileobj=buffer, mode="r:gz") as tar:
        members = [m for m in tar.getmembers() if m.name.endswith(".json")]
        print(f"   Found {len(members)} JSON chunk files")

        for member in tqdm(members, desc="Parsing chunks"):
            f = tar.extractfile(member)
            if not f:
                continue
            data = json.load(f)
            for product in data.get("products", []):
                cpe = product.get("cpe", {})
                cpe_name = cpe.get("cpeName", "")
                titles = cpe.get("titles", [])
                title = titles[0].get("title", "") if titles else ""
                if cpe_name:
                    records.append({
                        "cpe_name": cpe_name,
                        "title": title,
                        "deprecated": cpe.get("deprecated", False),
                    })
    return records

cpe_records = download_and_extract_cpe_dictionary(MIRROR_URL)
print(f"\n🎉 Total CPE records: {len(cpe_records):,}")

# DEBUG
print("\n" + "="*60)
print("DEBUG — Stage 1: Raw CPE Records")
print("="*60)
print(f"Type: {type(cpe_records)}")
print(f"Length: {len(cpe_records):,}")
print(f"First element: {cpe_records[0]}")
print(f"Keys: {list(cpe_records[0].keys())}")


⬇️  Downloading from: https://mirror.cveb.in/nvd/json/cpe/2.0/nvdcpe-2.0.tar.gz


✅ Downloaded: 72.5 MB
📦 Extracting tar.gz and parsing JSON chunks...
   Found 16 JSON chunk files


Parsing chunks:   0%|          | 0/16 [00:00<?, ?it/s]


🎉 Total CPE records: 1,637,758

DEBUG — Stage 1: Raw CPE Records
Type: <class 'list'>
Length: 1,637,758
First element: {'cpe_name': 'cpe:2.3:a:3com:3cdaemon:-:*:*:*:*:*:*:*', 'title': '3Com 3CDaemon', 'deprecated': False}
Keys: ['cpe_name', 'title', 'deprecated']


In [ ]:


def build_embed_text(cpe_name: str, title: str) -> str:
    parts = cpe_name.split(":")
    vendor = parts[3].replace("_", " ") if len(parts) > 3 else ""
    product = parts[4].replace("_", " ") if len(parts) > 4 else ""
    return f"{vendor} {product} {title}".strip()

rows = []
for rec in tqdm(cpe_records, desc="Building triples"):
    rows.append({
        "cpe_name": rec["cpe_name"],
        "title": rec["title"],
        "embed_text": build_embed_text(rec["cpe_name"], rec["title"]),
        "deprecated": rec["deprecated"],
    })

# Free raw records from RAM immediately
del cpe_records
gc.collect()

df = pd.DataFrame(rows)
del rows
gc.collect()

print(f"\nDataFrame built: {df.shape}")
print(f"   Columns: {list(df.columns)}")
print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

# Filter deprecated
print(f"\nFiltering deprecated CPEs...")
print(f"   Before: {len(df):,} rows")
df = df[df["deprecated"] == False].reset_index(drop=True)
print(f"   After:  {len(df):,} rows")
print(f"   Removed: {len(df) - len(df):,} wait...")

# DEBUG
print("\n" + "="*60)
print("DEBUG — Stage 2: DataFrame after filter")
print("="*60)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Dtypes:\n{df.dtypes}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())
print(f"\nSample embed_text:")
for i in range(3):
    print(f"  [{i}] {df['embed_text'].iloc[i][:80]}")


Building triples:   0%|          | 0/1637758 [00:00<?, ?it/s]


📊 DataFrame built: (1637758, 4)
   Columns: ['cpe_name', 'title', 'embed_text', 'deprecated']
   Memory: 463.4 MB

🗑️  Filtering deprecated CPEs...
   Before: 1,637,758 rows
   After:  1,544,118 rows
   Removed: 0 wait...

DEBUG — Stage 2: DataFrame after filter
Shape: (1544118, 4)
Columns: ['cpe_name', 'title', 'embed_text', 'deprecated']
Dtypes:
cpe_name      object
title         object
embed_text    object
deprecated      bool
dtype: object

First 3 rows:
                                         cpe_name                                             title                                                         embed_text  deprecated
0         cpe:2.3:a:3com:3cdaemon:-:*:*:*:*:*:*:*                                     3Com 3CDaemon                                        3com 3cdaemon 3Com 3CDaemon       False
1      cpe:2.3:h:3com:3crwe454g72:-:*:*:*:*:*:*:*  3Com 3Com OfficeConnect Wireless11g Access Point  3com 3crwe454g72 3Com 3Com OfficeConnect Wireless11g Access Point       False

In [ ]:


print(f"Loading model: {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU RAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n Model ready")


⏳ Loading model: all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🖥️  Device: cuda
   GPU: Tesla T4
   GPU RAM: 15.6 GB

✅ Model ready


In [ ]:

# This is the CRITICAL cell. We process 20K CPEs at a time,
# embed them, write to Parquet, then FREE the memory.
# Peak RAM stays under ~6 GB instead of crashing at 12+ GB.

# Define PyArrow schema once
schema = pa.schema([
    ("cpe_name", pa.string()),
    ("title", pa.string()),
    ("embed_text", pa.string()),
    ("embedding", pa.list_(pa.float32(), 384)),
])

writer = pq.ParquetWriter(OUTPUT_FILE, schema)
total_rows = len(df)
num_chunks = (total_rows + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"Starting chunked embedding")
print(f"   Total CPEs: {total_rows:,}")
print(f"   Chunk size: {CHUNK_SIZE:,}")
print(f"   Total chunks: {num_chunks}")
print(f"   Output: {OUTPUT_FILE}")
print()

for chunk_idx in range(num_chunks):
    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, total_rows)

    print(f"\nChunk {chunk_idx + 1}/{num_chunks} | Rows {start:,} → {end:,}")

    # Slice DataFrame (this is a view, not a copy — cheap)
    chunk_df = df.iloc[start:end]
    texts = chunk_df["embed_text"].tolist()

    # Embed this chunk only
    print(f"   Embedding {len(texts):,} texts...")
    chunk_embeddings = model.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        device=device,
    )
    print(f"   Embeddings shape: {chunk_embeddings.shape}")
    print(f"   Embeddings RAM: {chunk_embeddings.nbytes / 1024 / 1024:.1f} MB")

    # Build PyArrow table for this chunk
    print(f"   Building PyArrow table...")
    table = pa.table({
        "cpe_name": chunk_df["cpe_name"].tolist(),
        "title": chunk_df["title"].tolist(),
        "embed_text": texts,
        "embedding": chunk_embeddings.tolist(),
    }, schema=schema)

    # Write to Parquet
    print(f"   Writing to Parquet...")
    writer.write_table(table)

    # FREE EVERYTHING before next chunk
    del chunk_df, texts, chunk_embeddings, table
    gc.collect()

    # Show RAM usage
    import psutil
    ram_mb = psutil.virtual_memory().used / 1024 / 1024
    ram_pct = psutil.virtual_memory().percent
    print(f"   ✅ Done. RAM used: {ram_mb:.0f} MB ({ram_pct:.0f}%)")

writer.close()
print("\n" + "="*60)
print("ALL CHUNKS WRITTEN")
print("="*60)

# Show final file size
file_size = os.path.getsize(OUTPUT_FILE)
print(f"File: {OUTPUT_FILE}")
print(f"Size: {file_size / 1024 / 1024:.1f} MB ({file_size / 1024 / 1024 / 1024:.2f} GB)")

# Free the big DataFrame too
del df
gc.collect()


🚀 Starting chunked embedding
   Total CPEs: 1,544,118
   Chunk size: 20,000
   Total chunks: 78
   Output: cpe_embeddings.parquet


📦 Chunk 1/78 | Rows 0 → 20,000
   Embedding 20,000 texts...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

   Embeddings shape: (20000, 384)
   Embeddings RAM: 29.3 MB
   Building PyArrow table...
   Writing to Parquet...
   ✅ Done. RAM used: 3246 MB (28%)

📦 Chunk 2/78 | Rows 20,000 → 40,000
   Embedding 20,000 texts...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

   Embeddings shape: (20000, 384)
   Embeddings RAM: 29.3 MB
   Building PyArrow table...
   Writing to Parquet...
   ✅ Done. RAM used: 3308 MB (28%)

📦 Chunk 3/78 | Rows 40,000 → 60,000
   Embedding 20,000 texts...


...

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

   Embeddings shape: (20000, 384)
   Embeddings RAM: 29.3 MB
   Building PyArrow table...
   Writing to Parquet...
   ✅ Done. RAM used: 3353 MB (28%)

📦 Chunk 78/78 | Rows 1,540,000 → 1,544,118
   Embedding 4,118 texts...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

   Embeddings shape: (4118, 384)
   Embeddings RAM: 6.0 MB
   Building PyArrow table...
   Writing to Parquet...
   ✅ Done. RAM used: 3344 MB (28%)

🎉 ALL CHUNKS WRITTEN
File: cpe_embeddings.parquet
Size: 2353.8 MB (2.30 GB)


0

In [ ]:


print("Verifying Parquet file...")
df_check = pd.read_parquet(OUTPUT_FILE)

print(f"\nShape: {df_check.shape}")
print(f"Columns: {list(df_check.columns)}")
print(f"Dtypes:\n{df_check.dtypes}")
print()

print("First row:")
row = df_check.iloc[0]
print(f"  cpe_name:   {row['cpe_name']}")
print(f"  title:      {row['title']}")
print(f"  embed_text: {row['embed_text'][:70]}...")
print(f"  embedding:  [{', '.join(f'{x:.4f}' for x in row['embedding'][:5])}, ...]")
print(f"  embedding len: {len(row['embedding'])}")
print()

print("Vector norm checks:")
for i in [0, 100, 1000]:
    vec = np.array(df_check['embedding'].iloc[i])
    print(f"  Row {i}: norm={np.linalg.norm(vec):.6f}")

print(f"\nNull check:\n{df_check.isnull().sum()}")
print("\nParquet file is valid")


🔍 Verifying Parquet file...

Shape: (1544118, 4)
Columns: ['cpe_name', 'title', 'embed_text', 'embedding']
Dtypes:
cpe_name      object
title         object
embed_text    object
embedding     object
dtype: object

First row:
  cpe_name:   cpe:2.3:a:3com:3cdaemon:-:*:*:*:*:*:*:*
  title:      3Com 3CDaemon
  embed_text: 3com 3cdaemon 3Com 3CDaemon...
  embedding:  [-0.0959, -0.0499, -0.0805, -0.0666, -0.0368, ...]
  embedding len: 384

Vector norm checks:
  Row 0: norm=1.000000
  Row 100: norm=1.000000
  Row 1000: norm=1.000000

Null check:
cpe_name      0
title         0
embed_text    0
embedding     0
dtype: int64

✅ Parquet file is valid


In [ ]:

from google.colab import files
files.download(OUTPUT_FILE)
print("Download dialog should appear...")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download dialog should appear...


In [ ]:


print("Simulating semantic search...")

# Reload a sample for testing
df_sample = pd.read_parquet(OUTPUT_FILE).head(5000)

# Pick a target and find its neighbors
target_idx = 0
target_vec = np.array(df_sample['embedding'].iloc[target_idx])
target_text = df_sample['embed_text'].iloc[target_idx]

print(f"\nTarget: {df_sample['cpe_name'].iloc[target_idx]}")
print(f"Text:   {target_text}")
print()

# Brute-force cosine similarity on sample
sims = []
for i in range(len(df_sample)):
    vec = np.array(df_sample['embedding'].iloc[i])
    sim = np.dot(target_vec, vec) / (np.linalg.norm(target_vec) * np.linalg.norm(vec))
    sims.append((sim, i, df_sample['title'].iloc[i]))

sims.sort(reverse=True)

print("Top 5 matches:")
for sim, idx, title in sims[:5]:
    marker = " <-- TARGET" if idx == target_idx else ""
    print(f"  sim={sim:.4f} | {title[:55]}{marker}")

print("\nBottom 3 matches:")
for sim, idx, title in sims[-3:]:
    print(f"  sim={sim:.4f} | {title[:55]}")


🔍 Simulating semantic search...
